# 06 — Groupby y agregaciones

`groupby` es la operación más importante de pandas para análisis de datos. Divide el DataFrame en grupos, aplica una función a cada grupo, y combina los resultados.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
print(df.shape)


## Groupby básico

In [ ]:
# groupby devuelve un objeto GroupBy — no calcula nada hasta que se agrega
g = df.groupby('Region')
print(type(g))
print('Grupos:', list(g.groups.keys()))
print()

# Aplicar una función de agregación
print(g['Sales'].sum())
print()
print(g['Sales'].mean().round(2))


## agg() con lista de funciones

In [ ]:
# Aplicar múltiples funciones a la misma columna
resultado = (
    df.groupby('Category')['Sales']
    .agg(['sum', 'mean', 'count', 'min', 'max'])
    .round(2)
)
print(resultado)

# El resultado tiene MultiIndex en columnas si se agregan varias columnas


## agg() con agregaciones nombradas (tuplas)

La forma más profesional — produce columnas con nombres exactos sin necesidad de renombrar después.

In [ ]:
resultado = (
    df.groupby('Region')['Sales']
    .agg(
        revenue_total = 'sum',
        ticket_medio  = 'mean',
        num_pedidos   = 'count',
        venta_max     = 'max',
    )
    .round(2)
    .sort_values('revenue_total', ascending=False)
    .reset_index()
)
print(resultado)


## agg() con diccionario — distintas funciones por columna

In [ ]:
# Cuando se quieren funciones distintas para columnas distintas
resultado = df.groupby('Category').agg({
    'Sales':       ['sum', 'mean'],
    'Order ID':    'nunique',
    'Customer ID': 'nunique',
})

# El resultado tiene MultiIndex en columnas — aplanarlo
resultado.columns = ['_'.join(col) for col in resultado.columns]
resultado = resultado.reset_index()
print(resultado)


## Groupby por múltiples columnas

In [ ]:
resultado = (
    df.groupby(['Region', 'Category'])['Sales']
    .agg(revenue='sum', pedidos='count')
    .reset_index()
    .sort_values(['Region', 'revenue'], ascending=[True, False])
)
print(resultado)


## transform() — devuelve una Serie del mismo tamaño que el input

A diferencia de `agg()`, que reduce el número de filas, `transform()` devuelve un valor por cada fila del grupo. Útil para crear columnas basadas en estadísticas de grupo.

In [ ]:
# Añadir la media del grupo como columna nueva (sin reducir filas)
df['media_region'] = df.groupby('Region')['Sales'].transform('mean').round(2)

# Diferencia de cada venta respecto a la media de su región
df['desviacion_media'] = (df['Sales'] - df['media_region']).round(2)

print(df[['Region', 'Sales', 'media_region', 'desviacion_media']].head(10))


## pivot_table

Crea tablas cruzadas con filas, columnas, y valores. Más fácil de leer que un groupby multidimensional.

In [ ]:
tabla = pd.pivot_table(
    df,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum',
    margins=True,        # añade fila/columna de totales
    margins_name='Total'
).round(0)

print(tabla)


---
## Resumen

| Patrón | Sintaxis |
|--------|----------|
| Suma por grupo | `df.groupby('col')['val'].sum()` |
| Múltiples funciones | `.agg(['sum', 'mean', 'count'])` |
| Nombres exactos | `.agg(total='sum', media='mean')` |
| Distintas funciones por columna | `.agg({'col1': 'sum', 'col2': 'nunique'})` |
| Aplanar MultiIndex | `df.columns = ['_'.join(c) for c in df.columns]` |
| Sin reducir filas | `.transform('mean')` |
| Tabla cruzada | `pd.pivot_table(df, values, index, columns, aggfunc)` |
